# Example of SasRec training/inference

In [42]:
%load_ext autoreload
%autoreload 2	

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
SEED = 42

import os
import random
import numpy as np
import torch
import lightning as L

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

L.seed_everything(SEED, workers=True)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

# Для максимально строгой CPU-воспроизводимости. Медленнее, но честнее.
torch.set_num_threads(1)
torch.set_num_interop_threads(1)

import pandas as pd

Seed set to 42


## Preparing data
In this example, we will be using the MovieLens dataset, namely the 1m subset. It's demonstrated a simple case, so only item ids will be used as model input.

**NOTE**

Current implementation of SasRec is able to handle item and interactions features. It does not take into account user features. 

In [2]:
interactions = pd.read_csv("./data/ml1m_ratings.dat", sep="\t", names=["user_id", "item_id","rating","timestamp"])
interactions = interactions.drop(columns=["rating"])
item_features = (
    pd.read_csv("./data/ml1m_items.dat", sep="\t", names=["item_id","title","genre"])
    # .drop(columns=["release_year"])
    # .rename(columns={"class": "genre"})
)
item_features

,item_id,title,genre
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
3878,3948,Meet the Parents (2000),Comedy
3879,3949,Requiem for a Dream (2000),Drama
3880,3950,Tigerland (2000),Drama
3881,3951,Two Family House (2000),Drama


In [3]:
interactions["timestamp"] = interactions["timestamp"].astype("int64")
interactions = interactions.sort_values(by="timestamp")
interactions["timestamp"] = interactions.groupby("user_id").cumcount()
interactions

,user_id,item_id,timestamp
1000138,6040,858,0
1000153,6040,2384,1
999873,6040,593,2
1000192,6040,2019,3
1000007,6040,1961,4
...,...,...,...
825793,4958,2399,446
825438,4958,1407,447
825731,4958,2634,448
825724,4958,3264,449


### Encode catagorical data.
To ensure all categorical data is fit for training, it needs to be encoded using the `LabelEncoder` class. Create an instance of the encoder, providing a `LabelEncodingRule` for each categorcial column in the dataset that will be used in model. Note that ids of users and ids of items are always used.

In [4]:
interactions

,user_id,item_id,timestamp
1000138,6040,858,0
1000153,6040,2384,1
999873,6040,593,2
1000192,6040,2019,3
1000007,6040,1961,4
...,...,...,...
825793,4958,2399,446
825438,4958,1407,447
825731,4958,2634,448
825724,4958,3264,449


In [5]:
from replay.preprocessing import LabelEncoder, LabelEncodingRule

encoder = LabelEncoder(
    [
        # LabelEncodingRule("user_id"),
        LabelEncodingRule("item_id", handle_unknown="use_default_value", default_value="last"),
    ]
)
encoded_interactions = encoder.fit_transform(item_features)
encoded_interactions

,title,genre,item_id
0,Toy Story (1995),Animation|Children's|Comedy,0
1,Jumanji (1995),Adventure|Children's|Fantasy,1
2,Grumpier Old Men (1995),Comedy|Romance,2
3,Waiting to Exhale (1995),Comedy|Drama,3
4,Father of the Bride Part II (1995),Comedy,4
...,...,...,...
3878,Meet the Parents (2000),Comedy,3878
3879,Requiem for a Dream (2000),Drama,3879
3880,Tigerland (2000),Drama,3880
3881,Two Family House (2000),Drama,3881


In [6]:
encoded_interactions = encoder.transform(interactions)

In [7]:
# from replay.preprocessing import LabelEncoder, LabelEncodingRule

# encoder = LabelEncoder(
#     [
#         LabelEncodingRule("user_id"),
#         LabelEncodingRule("item_id", handle_unknown="use_default_value", default_value="last"),
#     ]
# )
# encoded_interactions = encoder.fit_transform(interactions)
# encoded_interactions

In [8]:
interactions[["item_id"]].nunique()

item_id    3706
dtype: int64

In [9]:
item_features[["item_id"]].nunique()

item_id    3883
dtype: int64

In [10]:
item_features[["item_id"]].min()

item_id    1
dtype: int64

In [11]:
len(encoder.mapping["item_id"])

3883

### Split interactions into the train, validation and test datasets using LastNSplitter
We use widespread splitting strategy Last-One-Out. We filter out cold items and users for simplicity.

In [12]:
from replay.splitters import LastNSplitter

splitter = LastNSplitter(
    N=1,
    divide_column="user_id",
    query_column="user_id",
    strategy="interactions",
    # drop_cold_users=True,
    # drop_cold_items=True
)

test_events, test_gt = splitter.split(encoded_interactions)
validation_events, validation_gt = splitter.split(test_events)
train_events = validation_events

### Dataset preprocessing ("baking")
SasRec expects each user in the batch to provide their events in form of a sequence. For this reason, the event splits must be properly processed using the `groupby_sequences` function provided by RePlay.

In [13]:
from replay.data.nn.utils import groupby_sequences


def bake_data(full_data):
    return groupby_sequences(events=full_data, groupby_col="user_id", sort_col="timestamp")

In [14]:
train_events = bake_data(train_events)

validation_events = bake_data(validation_events)
validation_gt = bake_data(validation_gt)

test_events = bake_data(test_events)
test_gt = bake_data(test_gt)

train_events

,user_id,timestamp,item_id
0,1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[3117, 1250, 1009, 1672, 2271, 1768, 3339, 118..."
1,2,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1180, 1192, 1199, 2648, 1273, 2874, 1207, 315..."
2,3,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[589, 2789, 1899, 3465, 1407, 1892, 1246, 1358..."
3,4,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1192, 1081, 3458, 476, 3399, 257, 1180, 1178,..."
4,5,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[2648, 907, 896, 352, 1230, 2119, 2789, 1111, ..."
...,...,...,...
6035,6036,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1672, 1814, 3369, 2307, 2359, 2503, 2423, 278..."
6036,6037,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[693, 1813, 3439, 1959, 1247, 558, 847, 3079, ..."
6037,6038,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[3327, 908, 1192, 2077, 1366, 352, 1063, 1132,..."
6038,6039,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[109, 279, 1998, 1211, 918, 3064, 935, 3019, 2..."


To ensure we don't have unknown users in ground truth, we join validation events and validation ground truth (also join test events and test ground truth correspondingly) by user ids to leave only the common ones.  

In [15]:
def add_gt_to_events(events_df, gt_df):
    gt_to_join = gt_df[["user_id", "item_id"]].rename(columns={"item_id": "ground_truth"})

    events_df = events_df.merge(gt_to_join, on="user_id", how="inner")
    return events_df

validation_events = add_gt_to_events(validation_events, validation_gt)
test_events = add_gt_to_events(test_events, test_gt)

In [16]:
from pathlib import Path

data_dir = Path("temp/data/")
data_dir.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = data_dir / "train.parquet"
VAL_PATH = data_dir / "val.parquet"
PREDICT_PATH = data_dir / "test.parquet"

ENCODER_PATH = data_dir / "encoder"

In [17]:
train_events.to_parquet(TRAIN_PATH)
validation_events.to_parquet(VAL_PATH)
test_events.to_parquet(PREDICT_PATH)

encoder.save(ENCODER_PATH)

/home/evtsinovnik/RePlay/replay/preprocessing/label_encoder.py:1002: UserWarning: There is already LabelEncoder object saved at the given path. File will be overwrited.
  warnings.warn(msg)
/home/evtsinovnik/RePlay/replay/preprocessing/label_encoder.py:563: UserWarning: There is already LabelEncodingRule object saved at the given path. File will be overwrited.
  warnings.warn(msg)


# Prepare to model training
### Create the tensor schema
A schema shows the correspondence of columns from the source dataset with the internal representation of tensors inside the model. It is required by the NN models to correctly create embeddings for every source column. Note that user_id does not required in `TensorSchema`.

Note that **cardinality** is the number of unique values ​in the item catalog (vocabulary). **Padding value** is the next value after the last one.

In [18]:
from replay.data import FeatureHint, FeatureType
from replay.data.nn import TensorFeatureInfo, TensorSchema


EMBEDDING_DIM = 300

encoder = encoder.load(ENCODER_PATH)
NUM_UNIQUE_ITEMS = len(encoder.mapping["item_id"])

tensor_schema = TensorSchema(
    [
        TensorFeatureInfo(
            name="item_id",
            is_seq=True,
            padding_value=NUM_UNIQUE_ITEMS,
            cardinality=NUM_UNIQUE_ITEMS,
            embedding_dim=EMBEDDING_DIM,
            feature_type=FeatureType.CATEGORICAL,
            feature_hint=FeatureHint.ITEM_ID,
        ),
    ]
)

### Configure ParquetModule and transformation pipelines

The `ParquetModule` class enables training of models on large datasets by reading data in batch-wise way. This class initialized with **paths to every data split, a metadata dict containing information about shape and padding value of every column and a dict of transforms**. `ParquetModule`'s  "transform pipelines" are stage-specific modules implementing additional preprocessing to be performed on batch level right before the forward pass.  

For SasRec model, RePlay provides a function that generates a sequence of appropriate transforms for each data split named **make_default_sasrec_transforms**.

Internally this function creates the following transforms:
1) Training:
    1. Create a target, which contains the shifted item sequence that represents the next item in the sequence (for the next item prediction task).
    2. Rename features to match it with expected format by the model during training.
    3. Unsqueeze target (*positive_labels*) and it's padding mask (*target_padding_mask*) for getting required shape of this tensors for loss computation.
    4. Group input features to be embed in expected format.

2) Validation/Inference:
    1. Rename/group features to match it with expected format by the model during valdiation/inference.

If a different set of transforms is required, you can create them yourself and submit them to the ParquetModule in the form of a dictionary where the key is the name of the split, and the value is the list of transforms. Available transforms are in the replay/nn/transforms/.

**Note:** One of the transforms for the training data prepares the initial sequence for the task of Next Item Prediction so it shifts the sequence of items. For the final sequence length to be correct, you need to set shape of item_id in metadata as **model sequence length + shift**. Default shift value is 1.

In [19]:
import torch



In [46]:
from replay.nn.transform import TargetMLMTransform, SequenceRollTransform, GroupTransform, CopyTransform, TokenMaskTransform, RenameTransform, UnsqueezeTransform
import copy

query_column = "user_id"
item_column = tensor_schema.item_id_feature_name
train_transforms = [
    RenameTransform({f"{item_column}_mask": "padding_mask"}),
    TokenMaskTransform(token_name="padding_mask"),
    CopyTransform({"item_id": "positive_labels"}),
    TargetMLMTransform(),
    UnsqueezeTransform("target_padding_mask", -1),
    UnsqueezeTransform("positive_labels", -1),
    GroupTransform({"feature_tensors": tensor_schema.names + ["token_mask"]}),
]

val_transforms = [
    RenameTransform({
        # query_column: "query_id", 
                     f"{item_column}_mask": "padding_mask"}),
    CopyTransform(mapping={"padding_mask": "token_mask"}),
    SequenceRollTransform("item_id", roll=-1, padding_value=tensor_schema["item_id"].padding_value),
    SequenceRollTransform("padding_mask", roll=-1, padding_value=1),
    SequenceRollTransform("token_mask", roll=-1, padding_value=0),
    CopyTransform({"item_id": "positive_labels"}),
    TargetMLMTransform(),
    UnsqueezeTransform("target_padding_mask", -1),
    UnsqueezeTransform("positive_labels", -1),
    GroupTransform({"feature_tensors": tensor_schema.names  + ["token_mask"]}),
]
test_transforms = copy.deepcopy(val_transforms)

predict_transforms = copy.deepcopy(val_transforms)

transforms = {
    "train": train_transforms,
    "validate": val_transforms,
    "test": test_transforms,
    "predict": predict_transforms,
}

In [47]:
from typing import Optional

MAX_SEQ_LEN = 100

def create_meta(shape: int, gt_shape: Optional[int] = None):
    meta = {
        "item_id": {"shape": shape, "padding": tensor_schema["item_id"].padding_value},
    }
    if gt_shape is not None:
        meta.update({"ground_truth": {"shape": gt_shape, "padding": -1}})

    return meta

train_metadata = {
    "train": create_meta(shape=MAX_SEQ_LEN),
    "validate": create_meta(shape=MAX_SEQ_LEN, gt_shape=1),
}

In [48]:
from replay.data.nn import ParquetModule

BATCH_SIZE = 512

parquet_module = ParquetModule(
    train_path=TRAIN_PATH,
    validate_path=VAL_PATH,
    batch_size=BATCH_SIZE,
    metadata=train_metadata,
    transforms=transforms,
)

/tmp/ipykernel_47511/1645025099.py:5: UserWarning: The following dataset paths aren't provided: test,predict. Make sure to disable these stages in your Lightning Trainer configuration.
  parquet_module = ParquetModule(


In [49]:
# parquet_module.setup("train")
# batch = parquet_module.compiled_transforms["train"](next(iter(parquet_module.train_dataloader())))
# (~batch["feature_tensors"]["token_mask"] & batch["padding_mask"]).float().mean()

In [50]:
# parquet_module.setup("validate")
# batch = parquet_module.compiled_transforms["validate"](next(iter(parquet_module.val_dataloader())))

## Train model
### Create SasRec model instance and run the training stage using lightning
We may now train the model using the Lightning trainer class. 

RePlay's implementation of SasRec is designed in a modular, **block-based approach**. Instead of passing configuration parameters to the constructor, SasRec is now built by providing fully initialized components that makes the model more flexible and easier to extend.

#### Default Configuration

Default SasRec model may be created quickly via method **from_params**. Default model instance has CE loss, original SasRec transformer layes, and embeddings are aggregated via sum.

In [51]:
from replay.nn.agg import SumAggregator
from replay.nn.embedding import SequenceEmbedding
from replay.nn.loss import CE

In [52]:
from replay.nn.sequential.bert4rec import Bert4Rec, Bert4RecBody, Bert4RecTransformerLayer, BertPositionAwareAggregator


NUM_BLOCKS = 2
NUM_HEADS = 4
DROPOUT = 0.5


# excluded_features = [
#     schema.query_id_feature_name,
#     schema.timestamp_feature_name,
#     *(excluded_features or []),
# ]
# excluded_features = list(set(excluded_features))

body = Bert4RecBody(
    embedder=SequenceEmbedding(tensor_schema
    ),
    embedding_aggregator=BertPositionAwareAggregator(
        embedding_aggregator=SumAggregator(embedding_dim=EMBEDDING_DIM),
        max_sequence_length=MAX_SEQ_LEN,
        dropout=DROPOUT,
    ),
    encoder=Bert4RecTransformerLayer(
        embedding_dim=EMBEDDING_DIM,
        num_heads=NUM_HEADS,
        num_blocks=NUM_BLOCKS,
        dropout=DROPOUT,
        feed_forward_hidden=4*EMBEDDING_DIM
        # activation="relu",
    ),
)


class Bert4RecWithBias(Bert4Rec):
    def __init__(self, body, loss, num_items: int):
        super().__init__(body=body, loss=loss)
        self.out_bias = torch.nn.Parameter(torch.empty(num_items))
        torch.nn.init.normal_(self.out_bias, mean=0.0, std=0.01)
        self.loss.logits_callback = self.get_logits

    def get_logits(self, model_embeddings, candidates_to_score=None):
        logits = super().get_logits(model_embeddings, candidates_to_score)

        if candidates_to_score is None:
            return logits + self.out_bias

        return logits + self.out_bias[candidates_to_score]

bert = (
    Bert4Rec(
    # Bert4RecWithBias(    
    body=body,
    loss=CE(ignore_index=tensor_schema.item_id_features.item().padding_value),
    # num_items=NUM_UNIQUE_ITEMS
))


In [53]:
# # new
# new_batch = parquet_module.compiled_transforms["train"](next(iter(parquet_module.train_dataloader())))
# bert.train()
# new_out = bert(
#     feature_tensors=new_batch["feature_tensors"],
#     padding_mask=new_batch["padding_mask"],
#     positive_labels=new_batch["positive_labels"],
#     negative_labels=None,
#     target_padding_mask=new_batch["target_padding_mask"],
# )
# new_loss = new_out["loss"]

# new_loss.item()


In [54]:
import contextlib

# def reset_like_legacy_bert4rec(model, seed: int = 42):
#     torch.manual_seed(seed)

#     # bias tied-head как в старом EmbeddingTyingHead
#     if hasattr(model, "out_bias"):
#         torch.nn.init.normal_(model.out_bias.data, mean=0.0, std=0.01)

#     for name, param in model.named_parameters():
#         if name.endswith("out_bias"):
#             continue
#         with contextlib.suppress(ValueError):
#             torch.nn.init.xavier_normal_(param.data)

# reset_like_legacy_bert4rec(bert, SEED)

A universal PyTorch Lightning module is provided. It can work with any NN model.

In [55]:
from replay.nn.lightning import LightningModule
from replay.nn.lightning.optimizer import OptimizerFactory

model = LightningModule(bert, optimizer_factory=OptimizerFactory(learning_rate=0.001))

To facilitate training, we add the following callbacks:
1) `ModelCheckpoint` - to save the best trained model based on its Recall metric. It's a default Lightning Callback.
1) `ComputeMetricsCallback` - to display a detailed validation metric matrix after each epoch. It's a custom RePlay callback for computing recsys metrics on validation and test stages.


In [56]:
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

from replay.nn.lightning.callback import ComputeMetricsCallback


checkpoint_callback = ModelCheckpoint(
    dirpath="bert4rec/.checkpoints2",
    save_top_k=1,
    verbose=True,
    monitor="recall@10",
    mode="max",
)

validation_metrics_callback = ComputeMetricsCallback(
    metrics=["map", "ndcg", "recall"],
    ks=[1, 5, 10, 20],
    item_count=NUM_UNIQUE_ITEMS,
)

csv_logger = CSVLogger(save_dir="sasrec/logs/train", name="SasRec-example")

trainer = L.Trainer(
    max_epochs=10,
    callbacks=[checkpoint_callback, validation_metrics_callback],
    logger=csv_logger,
    deterministic=True,
    benchmark=False,
)

trainer.fit(model, datamodule=parquet_module)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name  | Type     | Params | Mode 
-------------------------------------------
0 | model | Bert4Rec | 3.4 M  | train
-------------------------------------------
3.4 M     Trainable params
0         Non-trainable params
3.4 M     Total params
13.453    Total estimated model params size (MB)
42        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/evtsinovnik/RePlay/replay/nn/sequential/bert4rec/model.py:349: RuntimeWarning: Variable `positive_labels` is not None. This will have no effect at the inference stage.
  all(
/root/anaconda3/envs/bert-old/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (12) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

/home/evtsinovnik/RePlay/replay/nn/sequential/bert4rec/model.py:349: RuntimeWarning: Variable `positive_labels` is not None. This will have no effect at the inference stage.
  all(
Epoch 8, global step 108: 'recall@10' reached 0.02417 (best 0.02417), saving model to '/home/evtsinovnik/RePlay/examples/bert4rec/.checkpoints2/epoch=8-step=108.ckpt' as top 1


k             1         5         10        20
map     0.002318  0.005270  0.006892  0.008141
ndcg    0.002318  0.006927  0.010844  0.015514
recall  0.002318  0.012086  0.024172  0.042881 



Validation: |          | 0/? [00:00<?, ?it/s]

/home/evtsinovnik/RePlay/replay/nn/sequential/bert4rec/model.py:349: RuntimeWarning: Variable `positive_labels` is not None. This will have no effect at the inference stage.
  all(
Epoch 9, global step 120: 'recall@10' was not in top 1
`Trainer.fit` stopped: `max_epochs=10` reached.


k             1         5         10        20
map     0.002815  0.005304  0.006778  0.008291
ndcg    0.002815  0.006484  0.010159  0.015763
recall  0.002815  0.010099  0.021689  0.044040 



Now we can get the best model path stored in the checkpoint callback.

In [57]:
best_model_path = checkpoint_callback.best_model_path
best_model_path

'/home/evtsinovnik/RePlay/examples/bert4rec/.checkpoints2/epoch=8-step=108.ckpt'

## Inference

To obtain model scores, we will load the weights from the best checkpoint. To do this, we use the `LightningModule`, provide there the path to the checkpoint and the model instance.

In [58]:
# sasrec = SasRec.from_params(
#     schema=tensor_schema,
#     embedding_dim=EMBEDDING_DIM,
#     max_sequence_length=MAX_SEQ_LEN,
#     num_heads=NUM_HEADS,
#     num_blocks=NUM_BLOCKS,
#     dropout=DROPOUT,
#     excluded_features=None,
# )

best_model = LightningModule.load_from_checkpoint(best_model_path, model=bert)
best_model.eval();

Configure `ParquetModule` for inference

In [59]:
inference_metadata = {"predict": create_meta(shape=MAX_SEQ_LEN) | {"user_id": {}}}

parquet_module = ParquetModule(
    predict_path=PREDICT_PATH,
    batch_size=BATCH_SIZE,
    metadata=inference_metadata,
    transforms=transforms,
)

/tmp/ipykernel_47511/1878390203.py:3: UserWarning: The following dataset paths aren't provided: train,validate,test. Make sure to disable these stages in your Lightning Trainer configuration.
  parquet_module = ParquetModule(


During inference, we can use `TopItemsCallback`. Such callback allows you to get scores for each user throughout the entire catalog and get recommendations in the form of ids of items with the highest score values.


Recommendations can be fetched in four formats: PySpark DataFrame, Pandas DataFrame, Polars DataFrame or raw PyTorch tensors. Each of the types corresponds a callback. In this example, we'll be using the `PandasTopItemsCallback`.

In [60]:
from replay.nn.lightning.callback import PandasTopItemsCallback

csv_logger = CSVLogger(save_dir="sasrec/logs/test", name="SasRec-example")

TOPK = [1, 5, 10, 20, 100]

pandas_prediction_callback = PandasTopItemsCallback(
    top_k=max(TOPK),
    query_column="user_id",
    item_column="item_id",
    rating_column="score",
)

trainer = L.Trainer(callbacks=[pandas_prediction_callback], logger=csv_logger, inference_mode=True)
trainer.predict(best_model, datamodule=parquet_module, return_predictions=False)

pandas_res = pandas_prediction_callback.get_result()

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
/root/anaconda3/envs/bert-old/lib/python3.10/site-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.


Predicting: |          | 0/? [00:00<?, ?it/s]

/home/evtsinovnik/RePlay/replay/nn/sequential/bert4rec/model.py:349: RuntimeWarning: Variable `positive_labels` is not None. This will have no effect at the inference stage.
  all(


In [61]:
pandas_res

,user_id,item_id,score
0,1,2789,6.432658
0,1,1192,5.942246
0,1,1178,5.868874
0,1,585,5.809387
0,1,257,5.693648
...,...,...,...
6039,6040,3402,1.419595
6039,6040,1063,1.418045
6039,6040,1899,1.416173
6039,6040,2337,1.413984


### Calculating metrics

*test_gt* is already encoded, so we can use it for computing metrics.

In [62]:
from replay.metrics import MAP, OfflineMetrics, Precision, Recall
from replay.metrics.torch_metrics_builder import metrics_to_df

In [63]:
result_metrics = OfflineMetrics(
    [Recall(TOPK), Precision(TOPK), MAP(TOPK)],
    query_column="user_id",
    rating_column="score",
)(pandas_res, test_gt.explode("item_id"))

In [64]:
metrics_to_df(result_metrics)

k,1,5,10,20,100
MAP,0.003146,0.005516,0.006925,0.008110,0.010776
Precision,0.003146,0.002119,0.002136,0.001987,0.001679
Recall,0.003146,0.010596,0.021358,0.039735,0.167881


Let's call the `inverse_transform` encoder's function to get the final dataframe with recommendations

In [ ]:
encoder.inverse_transform(pandas_res)

,user_id,item_id,score
0,1,2858,5.992624
0,1,1196,5.665671
0,1,260,5.552308
0,1,593,5.535713
0,1,589,5.458629
...,...,...,...
6039,6040,2100,1.41113
6039,6040,2302,1.409827
6039,6040,1376,1.408515
6039,6040,3623,1.396935


: 

: 

: 

: 

: 

: 